# Medical Inventory Forecasting - Model Explainability

This notebook computes tree feature importances and SHAP values using `src.explainability`.


## 1. Import Modules and Load Data/Model


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path("..").resolve() if Path("..").joinpath("src").exists() else Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_processing import clean_product_data
from src.feature_engineering import prepare_ml_dataset, split_features_and_target, create_train_test_split
from src.model_persistence import load_model
from src.explainability import (
    get_feature_importance,
    plot_feature_importance,
    compute_shap_values,
    plot_shap_summary,
    plot_shap_waterfall
)

sns.set_theme(style="whitegrid")

product_data_path = REPO_ROOT / "data" / "Product_Level_Data_Final.csv"
stock = clean_product_data(pd.read_csv(product_data_path))
ml_model = prepare_ml_dataset(stock)
X, y = split_features_and_target(ml_model)
X_train, X_test, y_train, y_test = create_train_test_split(X, y, test_size=0.2, random_state=5)

model = load_model("models/medical_inventory_gb_model.pkl")


## 2. Feature Importance Analysis


In [ ]:
feat_imp = get_feature_importance(model, X_train.columns, top_n=10)
display(feat_imp)

fig_fi = plot_feature_importance(feat_imp, title="Gradient Boosting Feature Importance")
plt.show()


## 3. SHAP Explainability Plots


In [ ]:
explainer, shap_vals = compute_shap_values(model, X_test)

# SHAP Summary Plot
plot_shap_summary(shap_vals, X_test)

# SHAP Waterfall Plot for single prediction
plot_shap_waterfall(explainer, shap_vals, X_test, index=0)
